last modified date : 2026.03.15  
제작 : 박광석 (모두의연구소)

# 랭체인으로 RAG 시작하기

해당 노트는 Langchain으로 RAG를 구현하기 위해 필요한
각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다  




### Step 0 : 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [3]:
import os


In [4]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

In [5]:
#! curl ipinfo.io

In [6]:
from langchain_openai import ChatOpenAI

# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
)

In [7]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.
https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, 질의응답에 활용하고 싶은 PDF 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [8]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

In [9]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [10]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [11]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader

SV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, CSV 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

In [12]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/titanic.csv")

data = loader.load()

In [13]:
data[:3]

[Document(metadata={'source': '/content/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [14]:
from langchain_community.document_loaders import WebBaseLoader

In [15]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

#print(documents[0].page_content)

주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [16]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [17]:
with open("/content/state_of_the_union.txt") as f:
    text = f.read()

In [18]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [19]:
print(chunks[0])

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.


각 chunk의 길이를 확인해보겠습니다,

In [20]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]


### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [21]:
!pip install tiktoken

In [22]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [23]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]


글자 수와 토큰 수의 차이를 확인할 수 있습니다 !

### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
OpenAI 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

In [24]:
import openai

genai 라이브러리의 list_models 함수를 사용하여 사용 가능한 모델들의 목록을 가져옵니다.

In [25]:
client = openai.OpenAI()

In [26]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large


text-embedding-3-small은 가성비가 좋고, text-embedding-3-large는 성능이 더 강력합니다.

In [27]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

만약 여러분들이 Gemini를 사용하여 구축중이시라면, embedding은 지역에 따라 사용이 제한됩니다.  
주로 유럽권에서 제한되기 때문에, 다음 에러를 확인하신다면 Colab 파일의 서버 저장 위치를 확인 후, 다른 임베딩 모델로 변경해야합니다.  

Error embedding content: 400 User location is not supported for the API use.


In [28]:
#!curl ipinfo.io

In [29]:
# 400 User location is not supported for the API use 오류가 발생한다면, 이 블록을 대신 실행해주세요

# ! pip install -q sentence_transformers

#from langchain.embeddings import HuggingFaceEmbeddings
#embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embedding model 변수에 OpenAI 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [30]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [31]:
print(embeddings[1])

[0.006427764892578125, -0.020843505859375, -0.0216217041015625, 0.0235443115234375, -0.0428466796875, -0.004589080810546875, 0.039703369140625, 0.05267333984375, -0.0018796920776367188, -0.0237579345703125, 0.0138092041015625, 0.0159912109375, -0.044464111328125, -0.00011223554611206055, -0.007648468017578125, 0.02557373046875, 0.025421142578125, 0.03546142578125, -0.051788330078125, 0.01232147216796875, 0.02484130859375, -0.00047087669372558594, 0.0193939208984375, 0.07525634765625, -0.002044677734375, -0.009185791015625, 0.0168304443359375, -0.007129669189453125, 0.02655029296875, -0.04852294921875, 0.043182373046875, -0.042572021484375, 0.01163482666015625, -0.03289794921875, -0.0333251953125, -0.04620361328125, 0.0019054412841796875, 0.0225677490234375, -0.018096923828125, 0.03253173828125, 0.0293426513671875, 0.011749267578125, -0.01446533203125, 0.003063201904296875, 0.024322509765625, 0.04815673828125, -0.01354217529296875, 0.04937744140625, -0.0406494140625, 0.04400634765625, 0

In [32]:
len(embeddings[1])

1536

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [33]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [34]:
query = ["this is red fruit"]

In [35]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7478929665954975
0.4898791411166917
0.4083791543048118


빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [36]:
!pip install chromadb

In [37]:
!pip install langchain-chroma

In [38]:
from langchain_chroma import Chroma

In [39]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [40]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [41]:
#!pip show chromadb

Chroma에 임베딩 시킵니다  

In [42]:
db = Chroma.from_documents(docs, embedding_model)


이제 쿼리를 날려보겠습니다

In [43]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [44]:
print(docs[0].page_content)

DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [45]:
!pip install -U langchain langchain-classic

In [46]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [47]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-4o",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [48]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [49]:
query = "how demian looks like"
result = qa(query)

/tmp/ipykernel_9786/3336337621.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa(query)


Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His face has an almost feminine element and is different from the rest of the people around him. It is suggested that his appearance is unique, almost like an animal, a spirit, or an image, making him unimaginably different from others.

마크다운 형식으로 출력해봅니다

In [50]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His face has an almost feminine element and is different from the rest of the people around him. It is suggested that his appearance is unique, almost like an animal, a spirit, or an image, making him unimaginably different from others.

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [51]:
llm2 = ChatOpenAI(
    model="gpt-4o")
request = llm2.invoke("how demian looks like")
display(Markdown(request.content))


If you're referring to the character Demian from Hermann Hesse's novel "Demian," he doesn't have a definitive physical description, as Hesse focuses more on his personality and the philosophical ideas he represents. Demian is often depicted as having a striking presence, with a certain aura or charisma that draws people to him, reflecting his role as a guide or mentor to the protagonist, Emil Sinclair. Readers might imagine him as having an intense gaze, symbolic of his deep insight and understanding of the world. Beyond this, interpretations of his physical appearance can vary widely, allowing for personal imagination and artistic interpretation.

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
앞에서 진행한 내용으로, Demian을 다시 한번 읽어봅시다!  
완성하여 제출해주세요~


필요한 라이브러리를 모두 다운받습니다  
사실 %pip를 자주쓰는것 같음 아마 지금 가상환경 설정이 좀 이상해서 그런것 같기도 함 
조금만 꼬이면 !pip이 안돼서 %pip을 쓰는 일이 많은 것 같음.. 

In [10]:
%pip install -q langchain langchain-openai langchain-community langchain-core
%pip install -q langchain-text-splitters langchain-chroma chromadb
%pip install -q pypdf tiktoken

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 1 Document loader
우선 가상환경 파이썬 벚전이 헷갈려서 가상환경 위치를 확인함 
그리고 api키를 설정함 api키는 보안문제상 비워뒀음 

In [11]:
import sys
print(sys.executable)

c:\Users\akals\AppData\Local\Programs\Python\Python312\python.exe


In [ ]:
import os
os.environ['OPENAI_API_KEY'] = "여기에 api key 입력"

### 데이터로더 

랭체인의 기능중 하나인 PDF로더를 사용함 
코드는 간단한 로더에서는 내 파일 위치를 지정해주고 페이지에서는 로드와 함께 split()작업을 시작하는 듯 

In [13]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(r"C:\Users\akals\Downloads\RAG-아이펠\Demian.pdf")
pages = loader.load_and_split()

C:\Users\akals\AppData\Local\Temp\ipykernel_47868\1570776092.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\akals\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 전처리? 

청킹작업 자체가 전철리 작업과 굉장히 유사해 보였다 다른 점이라면 잘라낼 때마다 메타데이터라는 책갈피를 까워넣어 자료검색이 쉽게 한 점이다. 
TWO WOR.LDS~ Downloaded from https://www.holybooks.com 로 끝나는데 특이점은 추갈로 메타데이터를 만들어 자료에 대한 대략적인 
요약을 한다는 점이다. 
다만 메타데이터를 확인 했을 때는 불필요한 항목이 많아서 조금 의문이 들었다. 
producer, creator, moddate,source  이런 메타데이터는 굳이 필요없는게 아닌가? 

metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': 'C:\\Users\\akals\\Downloads\\RAG-아이펠\\Demian.pdf', 'total_pages': 182, 'page': 50, 'page_label': '51'}

In [14]:
print(len(pages))
print(pages[0])

182
page_content='DEMIAN 
• 
Downloaded from https://www.holybooks.com' metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': 'C:\\Users\\akals\\Downloads\\RAG-아이펠\\Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}


In [ ]:
print(pages[10])   
print("---")
print(pages[50])    

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

### 랭체인 스플리터 

추출된 텍스트에서 정한 파라미터 값 만큼의 기준으로 나누는 작업을 한다. 
여기가 진짜 청킹작업을 한다고 봐야할 것이다. 
len을 기준으로 자르고 1천자씩 그리고 앞의 내용을 100자 정도 겹치게 청킹작업을 한다. 

**파라미터 의미**
- `chunk_size=1000`: 한 chunk 최대 1000자
- `chunk_overlap=100`: 인접 chunk끼리 100자 겹침 (문장 끊김 방지)
- `length_function=len`: 길이 측정 기준은 *문자 수* (토큰 수로 바꿀 수도 있음)

**"Recursive"의 의미**
단순히 1000자마다 칼질하지 않고, 의미 단위를 우선 시도:
1. 빈 줄("\n\n", 문단 경계)로 자르려 시도
2. 안 맞으면 줄바꿈("\n")으로
3. 그래도 안 맞으면 공백(단어 경계)으로
4. 마지막으로 글자 단위 강제 자름

→ 큰 의미 단위(문단)부터 작은 단위(글자)로 *재귀적*으로 내려가며 자름.
→ 가능한 한 문장/문단이 잘리지 않게 보존.

**overlap의 진짜 목적**
chunk 경계에서 문장이 잘릴 때, 인접 chunk에 같은 텍스트를 100자 겹치게 둠. 
어느 chunk가 검색되어도 맥락이 살아있도록 안전장치 역할.

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
)

chunks = text_splitter.split_documents(pages)

print(f"총 chunk 개수: {len(chunks)}")
print("---")
print(chunks[0])

총 chunk 개수: 360
---
page_content='DEMIAN 
• 
Downloaded from https://www.holybooks.com' metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': 'C:\\Users\\akals\\Downloads\\RAG-아이펠\\Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}


In [21]:
print(chunks[20])
print("===")
print(chunks[21])

page_content='DEMIAN 
when he got up and turned to go home. When we were 
on the bridge I ventured timidly that I must go home. 
"No desperate hurry," Franz laughed. "We go the 
same way." ' 
He sauntered along slowly and I did not dare to go 
.ahead, but he was in fact going in the direction of our 
house. When we arrived, and I saw our front door and 
the fat doorknocker, the sun in the windows and the 
curtains in my mother's room, I breathed a sigh of 
relief. Back home I O good, blessed home-coming back 
to the world of light and peace I 
When I had quickly opened the door and slipped in 
ready to slam it behind me, Franz Kromer edged in too. 
In the cool, gloomy paved passage which was lit solely 
from the courtyard he stood close to me and said in a 
low voice, "No hurry, you I" 
I looked at him terrified. His grip on my arm was 
like a vice. I tried to guess what was going on in his 
mind and whether he was going to do me some mischief. 
If I were to let out a loud and vigorous

### 임베딩 시범 호출 

우선 임베딩이 제대로 작동하는지부터 확인해볼겸 테스트로 시험호출을 해본다. 
샘플 벡터는 이것은 테스트 문장입니다를 변환해 직접 값을 확인해본다. 
같은 차원을 가지고 있는지 값의 변환이 되는지를 확인해보고 바로 다음에서 직접 임베딩 변환을 한다. 

In [22]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

sample_vector = embeddings.embed_query("이것은 테스트 문장입니다")
print(f"벡터 차원: {len(sample_vector)}")
print(f"앞 5개 값: {sample_vector[:5]}")

벡터 차원: 1536
앞 5개 값: [4.202127456665039e-05, 0.032958984375, -0.00830841064453125, -0.01035308837890625, 0.0017490386962890625]


### 실제 임베딩 변환 후 테스트 

전체 임베딩을 넣어서 임베딩 변환을 진행 182개에서 260개로 늘어난 것이 특징 


In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_demian"  
)

print(f"저장된 문서 수: {vectorstore._collection.count()}")

저장된 문서 수: 360


In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}   
)

results = retriever.invoke("데미안은 어떤 인물인가?")

for i, doc in enumerate(results):
    print(f"--- 결과 {i+1} ---")
    print(doc.page_content[:200])   
    print(f"페이지: {doc.metadata.get('page')}")
    print()

--- 결과 1 ---
DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directe
페이지: 53

--- 결과 2 ---
DEMIAN 
character and have some significance. But I merely knew 
that Demian's mother was reported to be very wealthy. 
It was also said that neither she nor her son ever 
attended church. One boy won
페이지: 33

--- 결과 3 ---
DEMIAN 
dent. I could not forget him: And the worda he had said 
in that tavern in the suburbs came back to my mind, 
strangely fresh and si~ificant. "It is good to know that 
we have within us one wh
페이지: 93



In [27]:
%pip install -q langchain-ollama


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
# RAG 없이, LLM에게만 직접 묻기
direct_answer = llm.invoke("데미안은 어떤 인물인가?")
print(direct_answer.content)

'데미안(Demian)'은 헤르만 헤세(Hermann Hesse)의 동명 소설에 등장하는 핵심적인 인물입니다. 단순히 소설 속의 한 캐릭터를 넘어, **주인공의 자아 발견과 성장의 과정을 상징하는 매우 중요한 존재**로 해석됩니다.

따라서 데미안을 이해하려면, 그가 어떤 '사람'인지뿐만 아니라 그가 어떤 '상징'을 의미하는지 함께 이해하는 것이 중요합니다.

---

### 📚 1. 소설 속에서의 데미안 (The Character)

소설 속에서 데미안은 주인공 에밀 싱클레어(Emil Sinclair)의 삶에 나타나면서 그의 정신적 성장에 결정적인 영향을 미치는 인물입니다.

* **신비롭고 매력적인 존재:** 그는 주변 사람들에게 쉽게 설명할 수 없는 신비로운 분위기를 풍기며, 에밀에게 강한 끌림을 느끼게 합니다.
* **가르침의 전달자:** 그는 직접적으로 "이렇게 살아라"라고 가르치기보다는, 에밀이 스스로 진실을 발견하고 고통을 겪도록 유도하는 역할을 합니다.
* **경계인(Borderline):** 그는 사회의 규범이나 일반적인 가치관에 얽매이지 않는 자유로운 영혼을 상징합니다. 그는 주인공이 익숙한 세계를 벗어나 미지의 세계로 나아가도록 자극합니다.

### ✨ 2. 상징적 의미로서의 데미안 (The Symbol)

문학적, 철학적 관점에서 데미안은 단순한 친구나 스승이 아닙니다. 그는 주인공이 스스로 찾아야 할 **'진정한 자아'** 그 자체를 상징합니다.

#### ① 개성화(Individuation)의 상징
헤세의 철학에서 가장 중요한 개념 중 하나인 '개성화'는 개인이 사회가 정해준 틀이나 기대에서 벗어나 자신만의 고유한 존재로 완성되어 가는 과정을 의미합니다. 데미안은 바로 이 **'개성화의 길'**을 상징하는 안내자입니다.

#### ② 그림자 자아(Shadow Self)의 수용
데미안은 주인공이 외면하고 싶어 하거나, 사회적으로 용납되지 않는 자신의 어두운 면(욕망, 반항심, 혼란스러움)을 직면하게 만드는 거울과 같습니다. 그는 주인공이

In [ ]:
from langchain_ollama import ChatOllama
from langchain_classic.chains import RetrievalQA

llm = ChatOllama(
    model="gemma4:e4b",   
    temperature=0.0,
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,   
)

result = qa_chain.invoke({"query": "데미안은 어떤 인물인가?"})

print("=== 답변 ===")
print(result["result"])
print("\n=== 참조 페이지 ===")
for doc in result["source_documents"]:
    print(f"- page {doc.metadata.get('page')}")

=== 답변 ===
제공된 텍스트에 따르면, 데미안은 매우 **신비롭고, 깊이가 있으며, 사람들의 시선을 사로잡는 복합적인 인물**로 묘사됩니다.

그의 특징을 몇 가지 측면으로 나누어 설명할 수 있습니다.

### 1. 외모와 분위기 (신비로움)
*   **시간을 초월한 외모:** 그의 얼굴은 소년의 얼굴 같지도 않고, 남자의 얼굴 같지도 않으며, 심지어 거의 여성적인 요소가 있는 독특한 모습입니다.
*   **영원한 느낌:** 나이가 어리거나 늙었다고 규정할 수 없으며, 마치 **백 년을 살아온 듯한, 시대를 초월한** 느낌을 줍니다.
*   **태도:** 항상 우아하고 평온하며, 깊고 조용하지만 열정적인 몰입을 보여줍니다.

### 2. 지적 능력과 통찰력 (꿰뚫어 보는 힘)
*   **깊은 통찰:** 그는 화자(나)에 대해 "모든 것을 알고 있다"고 느껴지게 합니다.
*   **지혜로운 말:** 그가 했던 말들은 화자에게 오랫동안 의미심장하게 남아 영향을 미칩니다.

### 3. 사회적 배경과 평판 (강인함과 미스터리)
*   **신체적 강인함:** 그는 싸움을 걸어온 가장 강한 학생을 한 손으로 목덜미를 잡고 제압할 정도로 뛰어난 신체적 힘을 가지고 있습니다.
*   **배경의 미스터리:** 그의 어머니는 매우 부유하다고 알려져 있지만, 부자나 아들 모두 교회에 출석한 적이 없으며, 유대인이나 무슬림일 수 있다는 등 다양한 추측이 돌 정도로 배경이 비밀스럽습니다.
*   **현재 상황:** 그는 아마도 학업에 전념하고 있으며, 학창 시절이 끝나면 어머니와 고향을 떠날 예정인 것으로 보입니다.

**요약하자면,** 데미안은 단순히 한 명의 학생이라기보다는, **신비로운 아우라와 깊은 통찰력을 지닌, 시대를 초월한 듯한 매력을 가진 인물**로 묘사됩니다. 그의 존재 자체가 화자에게 큰 영감과 궁금증을 불러일으키는 핵심적인 존재입니다.

=== 참조 페이지 ===
- page 53
- page 33
- page 93


### 테스트 및 확인 

임베딩 모델에서는 제대로 RAG에서의 자료를 참조해 출력해주는 것을 볼 수가 있었다. 
모델은 젬마4: e4b 로 진행하였고 사실 이 모델이 작아 가장 좋은 차이를 볼수 있을것이라 생각했다. gpt모델이 아무래도 
성능이 좋을거라 생각해서 명확한 비교를 보기위한 방법이었음 

실제로 데미안에 대해 두 대답 모두 자세한 설명이 있었지만 RAG내에서의 실제 참조값을 바탕으로 설명해주는것을 알수가 있었다. 